In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
INSTANCE_NAME = f"{CATALOG}-game"

In [ ]:
%pip install databricks-sdk --upgrade "psycopg[binary]>=3.0"

In [ ]:
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
INSTANCE_NAME = f"{CATALOG}-game"

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance
from databricks.sdk.errors import NotFound, ResourceDoesNotExist
import uuid
import time

import sys
sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()

try:
    existing = w.database.get_database_instance(INSTANCE_NAME)
    print(f"\u267b\ufe0f Found existing database instance: {existing.name} (state: {existing.state})")
    instance = existing
except (NotFound, ResourceDoesNotExist):
    print(f"Creating new database instance: {INSTANCE_NAME}")
    instance = w.database.create_database_instance(
        DatabaseInstance(
            name=INSTANCE_NAME,
            capacity="CU_1"
        )
    )
    print(f"Created database instance: {instance.name}")
    add(CATALOG, "databaseinstances", {"name": instance.name})

print("Waiting for database instance to be available...")
for _ in range(60):
    state = str(w.database.get_database_instance(INSTANCE_NAME).state)
    if state == "DatabaseInstanceState.AVAILABLE":
        print(f"\u2705 Instance ready: {state}")
        break
    time.sleep(10)
else:
    raise TimeoutError(f"Database instance {INSTANCE_NAME} did not become available within 10 minutes")

In [ ]:
import psycopg
from psycopg.errors import DuplicateDatabase

instance = w.database.get_database_instance(INSTANCE_NAME)
cred = w.database.generate_database_credential(
    request_id=str(uuid.uuid4()),
    instance_names=[INSTANCE_NAME]
)
username = w.current_user.me().user_name

conn_string = (
    f"host={instance.read_write_dns} dbname=postgres "
    f"user={username} password={cred.token} sslmode=require"
)

with psycopg.connect(conn_string) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print(f"\u2705 Connected: {cur.fetchone()[0][:60]}")

        try:
            cur.execute("CREATE DATABASE game")
            print("\u2705 Created database 'game'")
        except DuplicateDatabase:
            print("♻️ Database 'game' already exists, skipping creation")

print(f"\u2705 Lakebase instance ready: {instance.read_write_dns}")

In [ ]:
game_conn_string = (
    f"host={instance.read_write_dns} dbname=game "
    f"user={username} password={cred.token} sslmode=require"
)

with psycopg.connect(game_conn_string) as conn:
    conn.autocommit = True
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS quest_state (
                player_id TEXT NOT NULL,
                level INT NOT NULL,
                status TEXT DEFAULT 'locked',
                answer TEXT,
                correct BOOLEAN,
                started_at TIMESTAMP,
                completed_at TIMESTAMP,
                hints_used INT DEFAULT 0,
                score INT DEFAULT 0,
                PRIMARY KEY (player_id, level)
            )
        """)
        print("\u2705 Created table quest_state")

        cur.execute("""
            CREATE TABLE IF NOT EXISTS leaderboard (
                player_id TEXT PRIMARY KEY,
                player_name TEXT NOT NULL,
                total_score INT DEFAULT 0,
                levels_completed INT DEFAULT 0,
                total_hints_used INT DEFAULT 0,
                started_at TIMESTAMP,
                finished_at TIMESTAMP
            )
        """)
        print("\u2705 Created table leaderboard")

print(f"\n\u2705 Game Lakebase setup complete")
print(f"   Instance: {INSTANCE_NAME}")
print(f"   Database: game")
print(f"   Tables: quest_state, leaderboard")